# **트랜스포머와 LLM**
---

# **4. LLM**: 사전학습 모델을 가져다 쓰고, 고쳐 쓰기
실행 환경: **Google Colab T4 GPU 권장** (런타임 → 런타임 유형 변경 → T4 GPU)

| Lab | 주제 | 모델 (라이선스) |
|---|---|---|
| 4-1 |  인코더 미세조정: NSMC 감성 분류 (Hugging Face `Trainer`) | KoELECTRA-small-v3 (Apache-2.0) |
| 4-2 |   디코더 LLM 들여다보기: 구조 · 토큰 · 다음 토큰 확률 · 생성 | Qwen3-0.6B (Apache-2.0) |
| 4-3 |   리뷰 분석기: 채팅 템플릿 + 배치 추론 + JSON 검증 | Qwen3-0.6B |
| 4-4 |   LoRA 미세조정: 0.6B 모델을 T4 한 장으로 고쳐 쓰기 | Qwen3-0.6B + PEFT |

⚠ 이 노트북의 출력은 비어 있습니다. 사전학습 모델 다운로드가 필요하므로 Colab에서 실행해 결과를 확인하세요.

In [1]:
%pip install -q "transformers>=4.51" "peft>=0.13" accelerate datasets pydantic

In [2]:
%pip install -U torchao

In [3]:
import os, re, json, time, random, urllib.request
import numpy as np, pandas as pd, torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM,
                          DataCollatorWithPadding, TrainingArguments, Trainer, pipeline, set_seed)
set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| GPU:", torch.cuda.get_device_name(0) if device == "cuda" else "-")

# ▶ 실습 설정 — 모델·데이터 크기는 여기서만 바꾼다
ENC_ID = "monologg/koelectra-small-v3-discriminator"   # Lab 4-1 인코더 (약 14M 파라미터)
LLM_ID = "Qwen/Qwen3-0.6B"                              # Lab 4-2~4-4 디코더 LLM
N_TRAIN, N_TEST = 30000, 10000                          # Lab 4-1 (T4 기준 약 5분 / CPU면 3000, 1000으로)
N_EVAL, N_LORA = 200, 2000                              # Lab 4-3·4-4 평가 리뷰 수, LoRA 학습 리뷰 수

for fn in ["ratings_train.txt", "ratings_test.txt"]:
    if not os.path.exists(fn):
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/e9t/nsmc/master/{fn}", fn)
train_df = pd.read_csv("ratings_train.txt", sep="\t").dropna().sample(N_TRAIN, random_state=42)
test_df = pd.read_csv("ratings_test.txt", sep="\t").dropna().sample(N_TEST, random_state=42)
eval_df = test_df.iloc[:N_EVAL].reset_index(drop=True)            # Lab 4-3·4-4 공통 평가 세트
print(len(train_df), len(test_df), len(eval_df))

device: cuda | GPU: Tesla T4
30000 10000 200


---
## **4-1. 인코더 미세조정** — "사전학습 + 적은 학습"의 위력
9차시: TF-IDF 87.0% · BiLSTM 86.6% (15만 건, 처음부터 학습) → 오늘: **사전학습된 인코더**를 3만 건으로 미세조정

In [4]:
from datasets import Dataset
enc_tok = AutoTokenizer.from_pretrained(ENC_ID)
enc_model = AutoModelForSequenceClassification.from_pretrained(
    ENC_ID,
    num_labels=2,
    id2label={0: "부정", 1: "긍정"},
    label2id={"부정": 0, "긍정": 1})
print(f"파라미터 수: {sum(p.numel() for p in enc_model.parameters()):,}")
print(enc_tok.tokenize("이 영화 개꿀잼ㅋㅋ 배우들 연기 미쳤다"))      # WordPiece 서브워드

def to_ds(df):
    ds = Dataset.from_pandas(df[["document", "label"]].rename(columns={"label": "labels"}), preserve_index=False)
    return ds.map(lambda b: enc_tok(b["document"], truncation=True, max_length=128), batched=True, remove_columns=["document"])
ds_train, ds_test = to_ds(train_df), to_ds(test_df)

def compute_metrics(p):
    pred = p.predictions.argmax(-1)
    return {"accuracy": float((pred == p.label_ids).mean())}

args = TrainingArguments(
    output_dir="koelectra-nsmc",
    num_train_epochs=2,
    learning_rate=5e-5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=256,
    weight_decay=0.01,
    warmup_steps=int(0.1 * 2 * len(ds_train) / 64),          # 전체 스텝의 10% 워밍업 (transformers 4.x·5.x 공통)
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=42)
trainer = Trainer(model=enc_model,
                  args=args,
                  train_dataset=ds_train,
                  eval_dataset=ds_test,
                  processing_class=enc_tok,
                  data_collator=DataCollatorWithPadding(enc_tok),
                  compute_metrics=compute_metrics)
t0 = time.time(); trainer.train(); print(f"학습 시간: {time.time()-t0:.0f}초")
enc_result = trainer.evaluate(); enc_result

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: monologg/koelectra-small-v3-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
classifier.dense.weight                           | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your d

파라미터 수: 14,122,498
['이', '영화', '개', '##꿀', '##잼', '##ㅋㅋ', '배우', '##들', '연기', '미쳤', '##다']


Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,0.397998,0.380923,0.831900
2,0.322713,0.360484,0.845800


학습 시간: 96초


Training Loss,Validation Loss,Epoch,Accuracy
0.322713,0.360484,2,0.845800


{'eval_loss': 0.36048394441604614, 'eval_accuracy': 0.8458}

In [5]:
# 실무: 저장 → 파이프라인으로 서비스처럼 사용
trainer.save_model("koelectra-nsmc/final")
enc_tok.save_pretrained("koelectra-nsmc/final")
clf = pipeline("text-classification",
               model="koelectra-nsmc/final",
               device=0 if device == "cuda" else -1)
probe = ["재미 없는 건 아니고 진짜 최고",
         "진짜 최고 아니고 재미 없는 건",
         "처음엔 지루했는데 끝까지 보니 인생 영화",
         "처음엔 재밌었는데 끝까지 보니 시간 낭비",
         "배우 연기는 좋은데 스토리가 별로"]
pd.DataFrame([{"문장": s, "예측": r["label"], "확률": round(r["score"], 3)} for s, r in zip(probe, clf(probe))])

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

,문장,예측,확률
0,재미 없는 건 아니고 진짜 최고,긍정,0.982
1,진짜 최고 아니고 재미 없는 건,부정,0.963
2,처음엔 지루했는데 끝까지 보니 인생 영화,긍정,0.671
3,처음엔 재밌었는데 끝까지 보니 시간 낭비,부정,0.948
4,배우 연기는 좋은데 스토리가 별로,부정,0.971


---
## **4-2. 디코더 LLM 들여다보기**

In [6]:
tok = AutoTokenizer.from_pretrained(LLM_ID)
llm = AutoModelForCausalLM.from_pretrained(LLM_ID, dtype="auto", device_map="auto")
cfg = llm.config
spec = {"층 수 (블록)": cfg.num_hidden_layers,
        "hidden size C": cfg.hidden_size,
        "Q 헤드": cfg.num_attention_heads,
        "KV 헤드 (GQA)": cfg.num_key_value_heads,
        "FFN 중간 크기": cfg.intermediate_size,
        "어휘 크기": cfg.vocab_size,
        "최대 위치": cfg.max_position_embeddings,
        "RoPE θ": getattr(cfg, "rope_theta", None) or (getattr(cfg, "rope_parameters", None) or {}).get("rope_theta"),   # 4.x / 5.x 위치 차이
        "총 파라미터": f"{sum(p.numel() for p in llm.parameters()):,}"}
pd.Series(spec, name=LLM_ID).to_frame()

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

,Qwen/Qwen3-0.6B
층 수 (블록),28
hidden size C,1024
Q 헤드,16
KV 헤드 (GQA),8
FFN 중간 크기,3072
어휘 크기,151936
최대 위치,40960
RoPE θ,1000000
총 파라미터,"596,049,920"


In [7]:
print(llm.model.layers[0])      # q_proj·k_proj·v_proj·o_proj(어텐션), gate·up·down_proj(SwiGLU FFN), RMSNorm

Qwen3DecoderLayer(
  (self_attn): Qwen3Attention(
    (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
    (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
    (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
    (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
    (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
    (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
  )
  (mlp): Qwen3MLP(
    (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
    (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
    (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
  (post_attention_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
)


In [8]:
# 토큰화: 한국어 문장은 몇 개의 토큰이 될까?
s = "트랜스포머는 어텐션만으로 문장을 이해한다."
ids = tok(s)["input_ids"]
print(len(s), "글자 →", len(ids), "토큰:", [tok.decode([i]) for i in ids])

# 다음 토큰 확률 Top-10: LLM의 출력은 '어휘 전체에 대한 확률분포'다
prompt = "대한민국의 수도는"
inputs = tok(prompt, return_tensors="pt").to(llm.device)
with torch.no_grad():
    probs = torch.softmax(llm(**inputs).logits[0, -1].float(), -1)
top = torch.topk(probs, 10)
pd.DataFrame({"다음 토큰": [repr(tok.decode([i])) for i in top.indices.tolist()], "확률": top.values.cpu().numpy().round(4)})

23 글자 → 16 토큰: ['트', '랜', '스', '포', '머', '는', ' 어', '텐', '션', '만', '으로', ' 문', '장을', ' 이해', '한다', '.']


,다음 토큰,확률
0,' 무엇',0.0548
1,' 어디',0.0454
2,' 도',0.0354
3,' 어떤',0.0293
4,' ____',0.0276
5,' 제',0.0259
6,' 한국',0.0259
7,'____',0.0259
8,' �',0.0228
9,' 서울',0.0215


In [9]:
# 채팅 형식 생성 (Qwen3 권장 설정: 비사고 모드 temperature 0.7, top_p 0.8, top_k 20)
messages = [{"role": "system", "content": "당신은 대학 강의 조교입니다. 한국어로 3문장 이내로 답하세요."},
            {"role": "user", "content": "셀프 어텐션을 고등학생에게 설명해 줘."}]
text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
print(text)                                               # 실제로 모델에 들어가는 문자열 (특수 토큰 포함)
inputs = tok([text], return_tensors="pt").to(llm.device)
out = llm.generate(**inputs, max_new_tokens=200, do_sample=True, temperature=0.7, top_p=0.8, top_k=20)
print(tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True))

<|im_start|>system
당신은 대학 강의 조교입니다. 한국어로 3문장 이내로 답하세요.<|im_end|>
<|im_start|>user
셀프 어텐션을 고등학생에게 설명해 줘.<|im_end|>
<|im_start|>assistant
<think>

</think>


셀프 어텐션은 학습을 돕는 도구로, 자신이 잘하고 싶은 내용을 집중하는 것과 같은 것을 말합니다.


---
## **4-3. 리뷰 분석기** — LLM 호출 패턴
- ① 시스템 프롬프트로 형식 고정
- ② **배치** 추론 (왼쪽 패딩)
- ③ JSON 파싱 + **스키마 검증**
- ④ 실패율·속도까지 측정

In [10]:
from pydantic import BaseModel, ValidationError
from typing import Literal

class ReviewResult(BaseModel):
    sentiment: Literal["긍정", "부정"]
    reason: str

SYSTEM = ('영화 리뷰의 감성을 분석해 JSON 한 줄로만 답하세요. 형식: {"sentiment": "긍정" 또는 "부정", "reason": "10자 이내 근거"}')

def parse(raw):
    m = re.search(r"\{.*?\}", raw, flags=re.S)                     # 앞뒤 잡담이 붙어도 JSON만 추출
    if not m:
        return None
    try:
        return ReviewResult.model_validate_json(m.group(0))        # 스키마 검증 (값이 '긍정'/'부정'인지까지)
    except ValidationError:
        return None

tok.padding_side = "left"                                          # 생성 모델 배치 추론은 왼쪽 패딩
def analyze(reviews, batch_size=16, max_new_tokens=48):
    results = []
    for i in range(0, len(reviews), batch_size):
        texts = [tok.apply_chat_template([{"role": "system", "content": SYSTEM}, {"role": "user", "content": r}],
                                         tokenize=False,
                                         add_generation_prompt=True,
                                         enable_thinking=False)
                 for r in reviews[i:i+batch_size]]
        enc = tok(texts, return_tensors="pt", padding=True).to(llm.device)
        with torch.no_grad():
            out = llm.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
        raws = tok.batch_decode(out[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
        results += [(raw, parse(raw)) for raw in raws]
    return results

t0 = time.time()
res = analyze(eval_df.document.tolist())
elapsed = time.time() - t0
parsed = [p for _, p in res]
ok = [p is not None for p in parsed]
pred = [1 if (p and p.sentiment == "긍정") else 0 for p in parsed]
acc_zero = float(np.mean([pr == y for pr, y, o in zip(pred, eval_df.label, ok) if o])) if any(ok) else float("nan")
print(f"JSON 파싱 성공률 {np.mean(ok):.1%} | 성공한 것 중 정확도 {acc_zero:.1%} | 리뷰당 {elapsed/len(res)*1000:.0f} ms")
pd.DataFrame({"리뷰": eval_df.document[:5], "정답": eval_df.label[:5].map({0: "부정", 1: "긍정"}), "LLM 원문": [r for r, _ in res[:5]]})

JSON 파싱 성공률 97.5% | 성공한 것 중 정확도 64.6% | 리뷰당 343 ms


,리뷰,정답,LLM 원문
0,찐한 여운과 함께...내 인생 최고의 미드,긍정,"{""sentiment"": ""긍정"", ""reason"": ""친절하고 성공적인 감정""}"
1,기대 이상이다,긍정,"{""sentiment"": ""부정"", ""reason"": ""기대 이상"""
2,신날 것 없는 애니.,부정,"{""sentiment"": ""부정"", ""reason"": ""신날을 위한 애니가 없어 감..."
3,마지막까지 힘을잃지않은 드라마~~,긍정,"{""sentiment"": ""긍정"", ""reason"": ""마지막까지 힘을 잃지 않았다.""}"
4,정말아니다. 그냥 뒤끝이 불쾌하고 상황에 대처하는 언니와 순경을 보고 너무 비현실적...,부정,"{""sentiment"": ""부정"", ""reason"": ""상황에 대한 비현실적이고 화..."


---
## **4-4. LoRA 미세조정** — 큰 모델은 그대로 두고, 작은 행렬만 학습
$W' = W + \dfrac{\alpha}{r} BA,\quad B\in\mathbb{R}^{d\times r},\ A\in\mathbb{R}^{r\times k},\ r \ll d$ (Hu et al., 2022)

In [11]:
LABEL = {0: "부정", 1: "긍정"}
SYSTEM_CLS = "영화 리뷰의 감성을 '긍정' 또는 '부정' 한 단어로만 답하세요."
def prompt_text(review):
    return tok.apply_chat_template([{"role": "system", "content": SYSTEM_CLS}, {"role": "user", "content": review}],
                                   tokenize=False, add_generation_prompt=True, enable_thinking=False)

# 평가: 생성하지 않고 '긍정' vs '부정' 첫 토큰 확률만 비교 (빠르고, 형식 오류가 없다)
pos_id = tok("긍정", add_special_tokens=False).input_ids[0]
neg_id = tok("부정", add_special_tokens=False).input_ids[0]
assert pos_id != neg_id
@torch.no_grad()
def score_accuracy(model, df, bs=32):
    model.eval(); correct = 0
    for i in range(0, len(df), bs):
        chunk = df.iloc[i:i+bs]
        enc = tok([prompt_text(r) for r in chunk.document], return_tensors="pt", padding=True).to(model.device)
        logits = model(**enc).logits[:, -1]                               # 왼쪽 패딩 → 마지막 위치가 다음 토큰
        pred = (logits[:, pos_id] > logits[:, neg_id]).long().cpu().numpy()
        correct += int((pred == chunk.label.values).sum())
    return correct / len(df)

acc_before = score_accuracy(llm, eval_df)
print(f"LoRA 전 정확도: {acc_before:.1%}")

LoRA 전 정확도: 52.5%


In [12]:
from peft import LoraConfig, get_peft_model
lora_cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                      target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
peft_model = get_peft_model(llm, lora_cfg)
peft_model.print_trainable_parameters()                                  # 학습 파라미터는 전체의 약 1~2%

# 학습 데이터: 프롬프트 + 정답 한 단어, 손실은 정답 토큰에만 (프롬프트는 -100으로 무시)
lora_df = pd.read_csv("ratings_train.txt", sep="\t").dropna().drop(train_df.index).sample(N_LORA, random_state=0)
examples = []
for r, y in zip(lora_df.document, lora_df.label):
    p_ids = tok(prompt_text(r), add_special_tokens=False).input_ids
    a_ids = tok(LABEL[y] + tok.eos_token, add_special_tokens=False).input_ids
    examples.append((p_ids + a_ids, [-100] * len(p_ids) + a_ids))

def collate(batch):
    L = max(len(x) for x, _ in batch)
    ids = torch.full((len(batch), L), tok.pad_token_id); lab = torch.full((len(batch), L), -100); att = torch.zeros((len(batch), L), dtype=torch.long)
    for i, (x, y) in enumerate(batch):                                     # 학습은 오른쪽 패딩
        ids[i, :len(x)], lab[i, :len(y)], att[i, :len(x)] = torch.tensor(x), torch.tensor(y), 1
    return ids, att, lab

peft_model.train()
opt = torch.optim.AdamW([p for p in peft_model.parameters() if p.requires_grad], lr=2e-4)
loader = torch.utils.data.DataLoader(examples, batch_size=8, shuffle=True, collate_fn=collate)
t0 = time.time()
for step, (ids, att, lab) in enumerate(loader, 1):
    with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=(device == "cuda" and torch.cuda.is_bf16_supported())):
        loss = peft_model(input_ids=ids.to(device), attention_mask=att.to(device), labels=lab.to(device)).loss
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 50 == 0:
        print(f"step {step:>4}/{len(loader)}  loss {loss.item():.3f}  ({time.time()-t0:.0f}s)")

acc_after = score_accuracy(peft_model, eval_df)
print(f"정확도: LoRA 전 {acc_before:.1%} → LoRA 후 {acc_after:.1%}  (학습 {N_LORA:,}건, {time.time()-t0:.0f}초)")

trainable params: 10,092,544 || all params: 606,142,464 || trainable%: 1.6650
step   50/250  loss 0.275  (60s)
step  100/250  loss 0.352  (123s)
step  150/250  loss 0.117  (189s)
step  200/250  loss 0.285  (250s)
step  250/250  loss 0.119  (308s)
정확도: LoRA 전 52.5% → LoRA 후 77.0%  (학습 2,000건, 324초)


In [13]:
# 실무: 어댑터만 저장 (수십 MB) → 나중에 원본 모델 + 어댑터로 다시 불러오기, 또는 병합해 배포
peft_model.save_pretrained("qwen3-nsmc-lora")
print(sorted(os.listdir("qwen3-nsmc-lora")))
# merged = peft_model.merge_and_unload(); merged.save_pretrained("qwen3-nsmc-merged")   # 병합 배포 (선택)

['README.md', 'adapter_config.json', 'adapter_model.safetensors']


## **정리**: 같은 NSMC, 다섯 가지 방법
| 방법 | 학습 데이터 | 기록할 정확도 |
|---|---|---|
| TF-IDF + 로지스틱 회귀 (9차시) | 15만 건 | 87.0% |
| BiLSTM (9차시) | 15만 건 | 86.6% |
| KoELECTRA 미세조정 (코드 4-1) | 3만 건 | ____ |
| Qwen3-0.6B 제로샷 (코드 4-3) | 0건 | ____ |
| Qwen3-0.6B + LoRA (코드 4-4) | 2천 건 | ____ |